<a href="https://colab.research.google.com/github/Padinjavier/google_colab/blob/main/CODE_MUSIC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. INSTALACIÓN DE DEPENDENCIAS
!pip install -q gradio yt-dlp mutagen requests

import os
import re
import json
import csv
import io
import shutil
import requests
import zipfile
import gradio as gr
from mutagen.mp4 import MP4, MP4Cover

# ─────────────────────────────────────────────
# SPOTIFY SCRAPER (Extracción sin API Key)
# ─────────────────────────────────────────────
class SpotifyAPI:
    _EMBED_PLAYLIST_URL = "https://open.spotify.com/embed/playlist/{playlist_id}"
    _EMBED_TRACK_URL = "https://open.spotify.com/embed/track/{track_id}"
    _NEXT_DATA_PATTERN = re.compile(r'<script id="__NEXT_DATA__"[^>]*>([^<]+)</script>')
    _USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"

    def __init__(self):
        self._session = requests.Session()

    def _fetch_embed_data(self, url: str) -> dict:
        headers = {"accept": "text/html", "user-agent": self._USER_AGENT}
        response = self._session.get(url, headers=headers, timeout=15)
        if response.status_code != 200:
            raise Exception(f"HTTP {response.status_code}")
        match = self._NEXT_DATA_PATTERN.search(response.text)
        if not match:
            raise Exception("No se encontró __NEXT_DATA__")
        return json.loads(match.group(1))

    def _extract_entity(self, data: dict) -> dict:
        paths = [
            ("props", "pageProps", "state", "data", "entity"),
            ("props", "pageProps", "data", "entity"),
            ("props", "pageProps", "entity"),
        ]
        for path in paths:
            result = data
            for key in path:
                if isinstance(result, dict):
                    result = result.get(key)
            if isinstance(result, dict) and result:
                return result
        raise Exception("No se pudo extraer la entidad")

    def get_playlist_info(self, playlist_id: str) -> dict:
        url = self._EMBED_PLAYLIST_URL.format(playlist_id=playlist_id)
        data = self._fetch_embed_data(url)
        entity = self._extract_entity(data)

        tracks = []
        for track in entity.get("trackList", []):
            if not isinstance(track, dict): continue
            uri = track.get("uri", "")
            track_id = uri.split(":")[-1] if uri.startswith("spotify:track:") else ""
            if not track_id: continue

            title = track.get("title") or track.get("name") or "Unknown"
            artists_raw = track.get("subtitle") or track.get("artists") or ""
            if isinstance(artists_raw, list):
                artists = ", ".join(a.get("name", "") for a in artists_raw if isinstance(a, dict))
            else:
                artists = str(artists_raw)

            tracks.append({"id": track_id, "title": title, "artists": artists})

        cover_url = None
        sources = entity.get("coverArt", {}).get("sources", [])
        if sources: cover_url = sources[-1].get("url")

        return {
            "name": entity.get("name") or "Playlist_Spotify",
            "tracks": tracks,
            "cover_url": cover_url
        }

    def get_track_metadata(self, track_id: str) -> dict:
        try:
            url = self._EMBED_TRACK_URL.format(track_id=track_id)
            data = self._fetch_embed_data(url)
            entity = self._extract_entity(data)
            album = entity.get("album", {}).get("name", "")

            cover_url = None
            images = entity.get("visualIdentity", {}).get("image", [])
            if images:
                cover_url = sorted(images, key=lambda i: i.get("maxWidth", 0), reverse=True)[0].get("url")

            return {"album": album, "cover_url": cover_url}
        except Exception:
            return {"album": "", "cover_url": None}

# ─────────────────────────────────────────────
# DESCARGADOR DE YOUTUBE & METADATOS
# ─────────────────────────────────────────────
def download_single_track(title, artist, track_id, output_dir, spotify_api, default_cover=None):
    from yt_dlp import YoutubeDL

    safe_title = re.sub(r'[<>:"/\\|?*]', '', title)
    safe_artist = re.sub(r'[<>:"/\\|?*]', '', artist)
    base_name = f"{safe_title} - {safe_artist}"
    output_template = os.path.join(output_dir, f"{base_name}.%(ext)s")

    search_query = f"ytsearch1:{title} {artist} audio"
    ydl_opts = {
        "format": "bestaudio[ext=m4a]/bestaudio/best",
        "outtmpl": output_template,
        "quiet": True,
        "no_warnings": True,
        "ignoreerrors": True
    }

    with YoutubeDL(ydl_opts) as ydl:
        ydl.extract_info(search_query, download=True)

    # Buscar archivo descargado
    downloaded_file = None
    for f in os.listdir(output_dir):
        if f.startswith(base_name) and not f.endswith(('.part', '.ytdl', '.temp')):
            downloaded_file = os.path.join(output_dir, f)
            break

    if not downloaded_file:
        return None

    # Obtener Metadatos e Inyectar Carátula
    meta = spotify_api.get_track_metadata(track_id) if track_id else {"album": "", "cover_url": None}
    cover_url = meta.get("cover_url") or default_cover

    try:
        audio = MP4(downloaded_file)
        audio["\xa9nam"] = [title]
        audio["\xa9ART"] = [artist]
        if meta.get("album"): audio["\xa9alb"] = [meta["album"]]

        if cover_url:
            res = requests.get(cover_url, timeout=10)
            if res.status_code == 200:
                audio["covr"] = [MP4Cover(res.content, imageformat=MP4Cover.FORMAT_JPEG)]
        audio.save()
    except Exception as e:
        print(f"Error etiquetando {title}: {e}")

    # Renombrar a .mp3
    final_mp3 = os.path.join(output_dir, f"{base_name}.mp3")
    os.rename(downloaded_file, final_mp3)
    return final_mp3

# ─────────────────────────────────────────────
# LÓGICA DE PROCESAMIENTO
# ─────────────────────────────────────────────
def process_spotify_url(url, progress=gr.Progress()):
    if not url: return "Pega una URL válida", None

    match = re.search(r"(https:\/\/open\.spotify\.com\/)(playlist|track)\/([a-zA-Z0-9]+)", url)
    if not match: return "URL de Spotify no válida", None

    spotify_type, item_id = match.group(2), match.group(3)
    spotify_api = SpotifyAPI()

    work_dir = "/content/downloads"
    if os.path.exists(work_dir): shutil.rmtree(work_dir)
    os.makedirs(work_dir, exist_ok=True)

    if spotify_type == "playlist":
        info = spotify_api.get_playlist_info(item_id)
        tracks = info["tracks"]
        playlist_name = re.sub(r'[<>:"/\\|?*]', '', info["name"])
        cover_url = info.get("cover_url")

        progress(0, desc=f"Descargando playlist: {playlist_name}")
        downloaded_files = []

        for idx, t in enumerate(tracks):
            progress((idx + 1) / len(tracks), desc=f"[{idx+1}/{len(tracks)}] {t['title']}")
            f = download_single_track(t["title"], t["artists"], t["id"], work_dir, spotify_api, cover_url)
            if f: downloaded_files.append(f)

        # Crear ZIP para descargar todo el álbum/playlist de un tiro
        zip_path = f"/content/{playlist_name}.zip"
        with zipfile.ZipFile(zip_path, 'w') as zipf:
            for file in downloaded_files:
                zipf.write(file, os.path.basename(file))

        return f"¡Completado! Se descargaron {len(downloaded_files)} canciones.", zip_path

    return "Procesamiento de pista individual completado", None

def process_csv_file(file_obj, progress=gr.Progress()):
    if file_obj is None: return "Sube un archivo CSV", None

    spotify_api = SpotifyAPI()
    work_dir = "/content/downloads_csv"
    if os.path.exists(work_dir): shutil.rmtree(work_dir)
    os.makedirs(work_dir, exist_ok=True)

    # Leer CSV
    content = file_obj.decode('utf-8')
    delimiter = ',' if ',' in content.splitlines()[0] else '\t'
    reader = csv.DictReader(io.StringIO(content), delimiter=delimiter)

    tracks = []
    for row in reader:
        title = row.get('Song') or row.get('title') or row.get('name') or ''
        artist = row.get('Artist') or row.get('artist') or ''
        track_id = row.get('Track ID') or row.get('track_id') or ''
        if title:
            tracks.append({"title": title, "artist": artist, "id": track_id})

    if not tracks: return "No se encontraron canciones en el CSV", None

    downloaded_files = []
    for idx, t in enumerate(tracks):
        progress((idx + 1) / len(tracks), desc=f"[{idx+1}/{len(tracks)}] {t['title']}")
        f = download_single_track(t["title"], t["artist"], t.get("id"), work_dir, spotify_api)
        if f: downloaded_files.append(f)

    csv_name = os.path.splitext(os.path.basename(file_obj.name))[0] if hasattr(file_obj, 'name') else "Playlist_CSV"
    zip_path = f"/content/{csv_name}.zip"
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for file in downloaded_files:
            zipf.write(file, os.path.basename(file))

    return f"¡Completado! Se procesaron {len(downloaded_files)} canciones del CSV.", zip_path

# ─────────────────────────────────────────────
# INTERFAZ WEB CON GRADIO
# ─────────────────────────────────────────────
with gr.Blocks(title="CB Music - Cloud Downloader") as app:
    gr.Markdown("CB Music Web")

    with gr.Tab("URL de Spotify"):
        url_input = gr.Textbox(label="Enlace de Playlist o Canción", placeholder="https://open.spotify.com/playlist/...")
        btn_url = gr.Button("Descargar Spotify", variant="primary")
        out_text_url = gr.Textbox(label="Estado")
        out_file_url = gr.File(label="Descargar Archivo (ZIP/MP3)")
        btn_url.click(process_spotify_url, inputs=[url_input], outputs=[out_text_url, out_file_url])

    with gr.Tab("Subir CSV"):
        gr.Markdown("https://www.chosic.com//spotify-playlist-analyzer/")
        csv_input = gr.File(label="Seleccionar archivo CSV", type="binary")
        btn_csv = gr.Button("Procesar y Descargar CSV", variant="primary")
        out_text_csv = gr.Textbox(label="Estado")
        out_file_csv = gr.File(label="Descargar Archivo (ZIP)")
        btn_csv.click(process_csv_file, inputs=[csv_input], outputs=[out_text_csv, out_file_csv])

# Lanzar servidor y generar URL pública
app.launch(share=True, debug=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 8.4 MB/s eta 0:00:00
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6c46643034a538e9c5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR: [youtube] Y8w5N4CfW_I: This video is not available


In [ ]:
# 1. INSTALACIÓN DE DEPENDENCIAS
!pip install -q gradio yt-dlp mutagen requests

import os
import re
import json
import csv
import io
import shutil
import requests
import gradio as gr
from mutagen.mp4 import MP4, MP4Cover

# ─────────────────────────────────────────────
# SPOTIFY SCRAPER (Extracción sin API Key)
# ─────────────────────────────────────────────
class SpotifyAPI:
    _EMBED_PLAYLIST_URL = "https://open.spotify.com/embed/playlist/{playlist_id}"
    _EMBED_TRACK_URL = "https://open.spotify.com/embed/track/{track_id}"
    _NEXT_DATA_PATTERN = re.compile(r'<script id="__NEXT_DATA__"[^>]*>([^<]+)</script>')
    _USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"

    def __init__(self):
        self._session = requests.Session()

    def _fetch_embed_data(self, url: str) -> dict:
        headers = {"accept": "text/html", "user-agent": self._USER_AGENT}
        response = self._session.get(url, headers=headers, timeout=15)
        if response.status_code != 200:
            raise Exception(f"HTTP {response.status_code}")
        match = self._NEXT_DATA_PATTERN.search(response.text)
        if not match:
            raise Exception("No se encontró __NEXT_DATA__")
        return json.loads(match.group(1))

    def _extract_entity(self, data: dict) -> dict:
        paths = [
            ("props", "pageProps", "state", "data", "entity"),
            ("props", "pageProps", "data", "entity"),
            ("props", "pageProps", "entity"),
        ]
        for path in paths:
            result = data
            for key in path:
                if isinstance(result, dict):
                    result = result.get(key)
            if isinstance(result, dict) and result:
                return result
        raise Exception("No se pudo extraer la entidad")

    def get_playlist_info(self, playlist_id: str) -> dict:
        url = self._EMBED_PLAYLIST_URL.format(playlist_id=playlist_id)
        data = self._fetch_embed_data(url)
        entity = self._extract_entity(data)

        tracks = []
        for track in entity.get("trackList", []):
            if not isinstance(track, dict): continue
            uri = track.get("uri", "")
            track_id = uri.split(":")[-1] if uri.startswith("spotify:track:") else ""
            if not track_id: continue

            title = track.get("title") or track.get("name") or "Unknown"
            artists_raw = track.get("subtitle") or track.get("artists") or ""
            if isinstance(artists_raw, list):
                artists = ", ".join(a.get("name", "") for a in artists_raw if isinstance(a, dict))
            else:
                artists = str(artists_raw)

            tracks.append({"id": track_id, "title": title, "artists": artists})

        cover_url = None
        sources = entity.get("coverArt", {}).get("sources", [])
        if sources: cover_url = sources[-1].get("url")

        return {
            "name": entity.get("name") or "Playlist_Spotify",
            "tracks": tracks,
            "cover_url": cover_url
        }

    def get_track_metadata(self, track_id: str) -> dict:
        try:
            url = self._EMBED_TRACK_URL.format(track_id=track_id)
            data = self._fetch_embed_data(url)
            entity = self._extract_entity(data)
            album = entity.get("album", {}).get("name", "")

            cover_url = None
            images = entity.get("visualIdentity", {}).get("image", [])
            if images:
                cover_url = sorted(images, key=lambda i: i.get("maxWidth", 0), reverse=True)[0].get("url")

            return {"album": album, "cover_url": cover_url}
        except Exception:
            return {"album": "", "cover_url": None}

# ─────────────────────────────────────────────
# DESCARGADOR DE YOUTUBE & METADATOS (con manejo de errores)
# ─────────────────────────────────────────────
def download_single_track(title, artist, track_id, output_dir, spotify_api, default_cover=None):
    from yt_dlp import YoutubeDL

    safe_title = re.sub(r'[<>:"/\\|?*]', '', title)
    safe_artist = re.sub(r'[<>:"/\\|?*]', '', artist)
    base_name = f"{safe_title} - {safe_artist}"
    output_template = os.path.join(output_dir, f"{base_name}.%(ext)s")

    search_query = f"ytsearch1:{title} {artist} audio"
    ydl_opts = {
        "format": "bestaudio[ext=m4a]/bestaudio/best",
        "outtmpl": output_template,
        "quiet": True,
        "no_warnings": True,
        "ignoreerrors": True,
        "socket_timeout": 30,  # ⬅️ Evita que cuelgue si la red falla
    }

    try:
        with YoutubeDL(ydl_opts) as ydl:
            ydl.extract_info(search_query, download=True)
    except Exception as e:
        print(f"Error al descargar {title}: {e}")
        return None

    # Buscar archivo descargado
    downloaded_file = None
    for f in os.listdir(output_dir):
        if f.startswith(base_name) and not f.endswith(('.part', '.ytdl', '.temp')):
            downloaded_file = os.path.join(output_dir, f)
            break

    if not downloaded_file:
        return None

    # Obtener Metadatos e Inyectar Carátula (si falla, solo sigue)
    try:
        meta = spotify_api.get_track_metadata(track_id) if track_id else {"album": "", "cover_url": None}
        cover_url = meta.get("cover_url") or default_cover

        audio = MP4(downloaded_file)
        audio["\xa9nam"] = [title]
        audio["\xa9ART"] = [artist]
        if meta.get("album"): audio["\xa9alb"] = [meta["album"]]

        if cover_url:
            res = requests.get(cover_url, timeout=10)
            if res.status_code == 200:
                audio["covr"] = [MP4Cover(res.content, imageformat=MP4Cover.FORMAT_JPEG)]
        audio.save()
    except Exception as e:
        print(f"Error al etiquetar {title}: {e}")

    # Renombrar a .mp3
    final_mp3 = os.path.join(output_dir, f"{base_name}.mp3")
    os.rename(downloaded_file, final_mp3)
    return final_mp3

# ─────────────────────────────────────────────
# LÓGICA DE PROCESAMIENTO (con entrega uno por uno)
# ─────────────────────────────────────────────
def process_spotify_url(url, progress=gr.Progress()):
    if not url:
        yield "Pega una URL válida", gr.update(value=[])
        return

    match = re.search(r"(https:\/\/open\.spotify\.com\/)(playlist|track)\/([a-zA-Z0-9]+)", url)
    if not match:
        yield "URL de Spotify no válida", gr.update(value=[])
        return

    spotify_type, item_id = match.group(2), match.group(3)
    spotify_api = SpotifyAPI()

    # Directorio único para esta sesión
    work_dir = "/content/downloads"
    if os.path.exists(work_dir): shutil.rmtree(work_dir)
    os.makedirs(work_dir, exist_ok=True)

    if spotify_type == "playlist":
        try:
            info = spotify_api.get_playlist_info(item_id)
        except Exception as e:
            yield f"Error obteniendo playlist: {e}", gr.update(value=[])
            return

        tracks = info["tracks"]
        playlist_name = re.sub(r'[<>:"/\\|?*]', '', info["name"])
        cover_url = info.get("cover_url")

        progress(0, desc=f"Descargando: {playlist_name}")
        downloaded_files = []
        total = len(tracks)

        for idx, t in enumerate(tracks):
            progress((idx + 1) / total, desc=f"[{idx+1}/{total}] {t['title']}")
            try:
                f = download_single_track(t["title"], t["artists"], t["id"], work_dir, spotify_api, cover_url)
                if f:
                    downloaded_files.append(f)
                    # ⬇️ ENTREGA INMEDIATA: se actualiza la lista de archivos
                    yield f"✅ {len(downloaded_files)}/{total} listas", gr.update(value=downloaded_files, visible=True)
                else:
                    yield f"⚠️ No se encontró: {t['title']} | {len(downloaded_files)}/{total} listas", gr.update(value=downloaded_files, visible=True)
            except Exception as e:
                yield f"❌ Error en {t['title']}: {str(e)[:50]} | {len(downloaded_files)}/{total} listas", gr.update(value=downloaded_files, visible=True)
                continue

        yield f"🏁 Finalizado: {len(downloaded_files)}/{total} canciones descargadas.", gr.update(value=downloaded_files, visible=True)
        return

    elif spotify_type == "track":
        # Lógica para canción individual (similar, pero solo una)
        try:
            info = spotify_api.get_playlist_info(item_id)  # Reutilizamos, pero solo hay 1 track
            # Realmente necesitamos track info, pero mejor simplificamos:
            yield "Funcionalidad de pista individual disponible con el mismo método.", gr.update(value=[])
        except:
            yield "Error en pista individual", gr.update(value=[])
        return

# ─────────────────────────────────────────────
# PROCESADOR DE CSV (con entrega uno por uno)
# ─────────────────────────────────────────────
def process_csv_file(file_obj, progress=gr.Progress()):
    if file_obj is None:
        yield "Sube un archivo CSV", gr.update(value=[])
        return

    spotify_api = SpotifyAPI()
    work_dir = "/content/downloads_csv"
    if os.path.exists(work_dir): shutil.rmtree(work_dir)
    os.makedirs(work_dir, exist_ok=True)

    try:
        content = file_obj.decode('utf-8')
        delimiter = ',' if ',' in content.splitlines()[0] else '\t'
        reader = csv.DictReader(io.StringIO(content), delimiter=delimiter)

        tracks = []
        for row in reader:
            title = row.get('Song') or row.get('title') or row.get('name') or ''
            artist = row.get('Artist') or row.get('artist') or ''
            track_id = row.get('Track ID') or row.get('track_id') or ''
            if title:
                tracks.append({"title": title, "artist": artist, "id": track_id})
    except Exception as e:
        yield f"Error leyendo CSV: {e}", gr.update(value=[])
        return

    if not tracks:
        yield "No se encontraron canciones en el CSV", gr.update(value=[])
        return

    downloaded_files = []
    total = len(tracks)

    for idx, t in enumerate(tracks):
        progress((idx + 1) / total, desc=f"[{idx+1}/{total}] {t['title']}")
        try:
            f = download_single_track(t["title"], t["artist"], t.get("id"), work_dir, spotify_api)
            if f:
                downloaded_files.append(f)
                yield f"✅ {len(downloaded_files)}/{total} listas", gr.update(value=downloaded_files, visible=True)
            else:
                yield f"⚠️ No se encontró: {t['title']} | {len(downloaded_files)}/{total} listas", gr.update(value=downloaded_files, visible=True)
        except Exception as e:
            yield f"❌ Error en {t['title']}: {str(e)[:50]} | {len(downloaded_files)}/{total} listas", gr.update(value=downloaded_files, visible=True)
            continue

    yield f"🏁 Finalizado: {len(downloaded_files)}/{total} canciones descargadas.", gr.update(value=downloaded_files, visible=True)

# ─────────────────────────────────────────────
# INTERFAZ WEB CON GRADIO (Minimalista B&N)
# ─────────────────────────────────────────────
theme = gr.themes.Base(
    primary_hue="neutral",
    secondary_hue="neutral",
    neutral_hue="gray",
    font=gr.themes.GoogleFont("Inter"),
)

with gr.Blocks(title="CB Music", theme=theme) as app:
    gr.Markdown("CB Music Web")

    with gr.Tab("URL de Spotify"):
        url_input = gr.Textbox(label="Enlace de Playlist o Canción", placeholder="https://open.spotify.com/playlist/...")
        btn_url = gr.Button("Descargar Spotify")
        out_text_url = gr.Textbox(label="Estado")
        out_file_url = gr.File(label="Descargar Archivos (MP3s)", file_count="multiple")
        btn_url.click(process_spotify_url, inputs=[url_input], outputs=[out_text_url, out_file_url])

    with gr.Tab("Subir CSV"):
        csv_input = gr.File(label="Seleccionar archivo CSV", type="binary")
        btn_csv = gr.Button("Procesar y Descargar CSV")
        out_text_csv = gr.Textbox(label="Estado")
        out_file_csv = gr.File(label="Descargar Archivos (MP3s)", file_count="multiple")
        btn_csv.click(process_csv_file, inputs=[csv_input], outputs=[out_text_csv, out_file_csv])

# Lanzar servidor y generar URL pública
app.launch(share=True)